In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import xgboost as xgb
import re

df = pd.read_csv("data/lendingclub_clean.csv", low_memory=False)
df['emp_length'] = df['emp_length'].fillna('Missing')
df['home_ownership'] = df['home_ownership'].replace({'NONE': 'OTHER'})

df_mature = df[df['event_type'].isin([1, 2])].copy()
df_mature['default_flag'] = (df_mature['event_type'] == 1).astype(int)

print(f"Số khoản vay mature: {len(df_mature):,} (từ {len(df):,} tổng)")
print(f"Tỷ lệ default trong nhóm mature: {df_mature['default_flag'].mean():.2%}")

feature_cols = ['loan_amnt', 'int_rate', 'grade', 'term', 'annual_inc', 'dti',
                 'fico_range_low', 'home_ownership', 'purpose', 'verification_status',
                 'emp_length', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_util',
                 'application_type', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies']

model_df = df_mature[feature_cols + ['default_flag']].copy()
for col in ['revol_util', 'dti', 'mort_acc', 'pub_rec_bankruptcies']:
    model_df[col] = model_df[col].fillna(model_df[col].median())
model_df = model_df.dropna(subset=[c for c in feature_cols if c != 'emp_length'])

X = pd.get_dummies(model_df[feature_cols], drop_first=True)
y = model_df['default_flag']

def clean_feature_name(name):
    name = name.replace('< ', 'under_').replace('<', 'lt')
    name = re.sub(r'[\[\]]', '', name)
    name = name.replace(' ', '_')
    return name

X.columns = [clean_feature_name(c) for c in X.columns]
print(f"\nSố feature sau encode: {X.shape[1]}")

X_sample, _, y_sample, _ = train_test_split(X, y, train_size=300_000, 
                                              random_state=42, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample)

print(f"Train: {X_train.shape[0]:,}, Test: {X_test.shape[0]:,}")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, eval_metric='auc', random_state=42
)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"\nAUC: {auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"\n{classification_report(y_test, model.predict(X_test))}")

importance = pd.DataFrame({
    'feature': X.columns, 'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print("\n--- Top 15 feature importance ---")
print(importance.head(15).to_string(index=False))

model.save_model("data/xgb_default_model.json")
X.columns.to_series().to_csv("data/model_features.csv", index=False)
print("\nĐã lưu model và danh sách feature")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/lendingclub_clean.csv", low_memory=False)
df['emp_length'] = df['emp_length'].fillna('Missing')
df['home_ownership'] = df['home_ownership'].replace({'NONE': 'OTHER'})

df_mature = df[df['event_type'].isin([1, 2])].copy()
df_mature['default_flag'] = (df_mature['event_type'] == 1).astype(int)
df_mature['treatment'] = (df_mature['verification_status'] != 'Not Verified').astype(int)

print(f"Tỷ lệ được xác minh: {df_mature['treatment'].mean():.2%}")
print(f"\nChurn rate thô:")
print(f"  Nhóm được xác minh: {df_mature[df_mature['treatment']==1]['default_flag'].mean():.2%}")
print(f"  Nhóm không xác minh: {df_mature[df_mature['treatment']==0]['default_flag'].mean():.2%}")

import re
import xgboost as xgb
from sklearn.linear_model import LogisticRegression

feature_cols = ['loan_amnt', 'int_rate', 'grade', 'term', 'annual_inc', 'dti',
                 'fico_range_low', 'home_ownership', 'purpose',
                 'emp_length', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_util',
                 'application_type', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies']

model_df = df_mature[feature_cols + ['default_flag', 'treatment']].copy()
for col in ['revol_util', 'dti', 'mort_acc', 'pub_rec_bankruptcies']:
    model_df[col] = model_df[col].fillna(model_df[col].median())
model_df = model_df.dropna(subset=[c for c in feature_cols if c != 'emp_length'])

model_df = model_df.sample(n=300_000, random_state=42).reset_index(drop=True)

X = pd.get_dummies(model_df[feature_cols], drop_first=True)
def clean_name(name):
    name = name.replace('< ', 'under_').replace('<', 'lt')
    name = re.sub(r'[\[\]]', '', name)
    return name.replace(' ', '_')
X.columns = [clean_name(c) for c in X.columns]

y = model_df['default_flag']
treatment = model_df['treatment']

prop_model = LogisticRegression(max_iter=2000)
prop_model.fit(X, treatment)
propensity = prop_model.predict_proba(X)[:, 1]

print("--- Overlap check ---")
print(f"Propensity TB - treatment: {propensity[treatment==1].mean():.4f}")
print(f"Propensity TB - control: {propensity[treatment==0].mean():.4f}")
print(f"Range treatment: [{propensity[treatment==1].min():.4f}, {propensity[treatment==1].max():.4f}]")
print(f"Range control: [{propensity[treatment==0].min():.4f}, {propensity[treatment==0].max():.4f}]")

mask_t = treatment == 1
mask_c = treatment == 0

model_treated = xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
model_treated.fit(X[mask_t], y[mask_t])
model_control = xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
model_control.fit(X[mask_c], y[mask_c])

D_treated = y[mask_t] - model_control.predict_proba(X[mask_t])[:, 1]
D_control = model_treated.predict_proba(X[mask_c])[:, 1] - y[mask_c]

tau_model_t = xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
tau_model_t.fit(X[mask_t], D_treated)
tau_model_c = xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
tau_model_c.fit(X[mask_c], D_control)

g = propensity.clip(0.05, 0.95)
tau_t_full = tau_model_t.predict(X)
tau_c_full = tau_model_c.predict(X)

cate_raw = g * tau_c_full + (1 - g) * tau_t_full
model_df['CATE'] = -cate_raw

print(f"\n--- Phân phối CATE (dương = xác minh giúp giảm rủi ro) ---")
print(model_df['CATE'].describe())

print(f"\n--- ATE trung bình toàn dataset ---")
print(f"Tác động trung bình của việc xác minh lên xác suất vỡ nợ: {model_df['CATE'].mean():.4f}")
print(f"(So sánh thô KHÔNG điều chỉnh: {df_mature[df_mature['treatment']==1]['default_flag'].mean() - df_mature[df_mature['treatment']==0]['default_flag'].mean():.4f})")

model_df.to_csv("data/lendingclub_with_cate.csv", index=False)
print("\nĐã lưu data/lendingclub_with_cate.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("data/lendingclub_with_cate.csv")

print("--- CATE trung bình theo Grade (dương = xác minh giúp giảm rủi ro) ---")
cate_by_grade = df.groupby('grade')['CATE'].agg(['mean', 'std', 'count'])
print(cate_by_grade)

print("\n--- CATE trung bình theo mức DTI (chia tứ phân vị) ---")
df['dti_quartile'] = pd.qcut(df['dti'], q=4, labels=['Q1_thấp', 'Q2', 'Q3', 'Q4_cao'])
print(df.groupby('dti_quartile')['CATE'].mean())

print("\n--- Tỷ lệ khoản vay có CATE dương (xác minh THỰC SỰ có lợi) ---")
print(f"{(df['CATE'] > 0).mean():.2%} khoản vay có tác động dương từ xác minh")